In [3]:
import glob
import numpy
import pandas
import seaborn
import matplotlib.pyplot as plt
from tqdm import tqdm
import multiprocessing as mp
import os
from build import build_model

In [4]:
from IPython.display import display, HTML, Math, Markdown
display(HTML("<style>.container { width:95% !important; }</style>"))

%load_ext autoreload
%autoreload 2

In [5]:
# !bash report.sh
organisms = set(pandas.read_csv("biomass_constrained.txt",index_col=0,header=None).index.to_list())
done = {i.split(".csv")[0] for i in os.listdir("./cases/fluxes/micro_oxygen")}

In [6]:
run_for = organisms - done
len(run_for)

0

In [7]:
# protein
exchanges = ["EX_o2(e)"]

In [8]:
from diets import load_model,constrain_exchanges,get_bounds
from overflow import get_overflow
import coralme
import cobra

conditions = ["micro"]
metabolite = "oxygen"

for c in conditions:
    name = "{}_{}".format(c,metabolite)
    fluxdir = "./cases/fluxes/{}".format(name)
    overflowdir = "./cases/overflow/{}".format(name)
    if not os.path.isdir(fluxdir):
        os.makedirs(fluxdir)
        os.makedirs(overflowdir)

def run(org):
    model = coralme.io.pickle.load_pickle_me_model("./me-models/{}/MEModel-BIO-{}-ME-TS.pkl".format(org,org))
    m_model = cobra.io.json.load_json_model("./agora-models/AGORA_2_01_json_renamed/{}.json".format(org))

    if m_model.reactions.has_id("CYTBD"):
        rxn = m_model.reactions.get_by_id("CYTBD")
        if "Unknown" in rxn.gene_reaction_rule:
            for rxn in model.reactions.query("CYTBD"):
                rxn.bounds = (0,0)

    sol0 = model.solution
    df0 = sol0.to_frame()
    # Exchanges
    rlist = exchanges
    # Run for all conditions
    for condition in conditions:
        name = "{}_{}".format(condition,metabolite)
        dct = {k:-0.5 for k in rlist if k in model.reactions}
        constrain_exchanges(model,dct)
        model.optimize()
        model.solution.to_frame().to_csv("cases/fluxes/{}/{}.csv".format(name,org))
        get_overflow(model).to_csv("./cases/overflow/{}/{}.csv".format(name,org))
        

In [9]:
org = "Pseudomonas_aeruginosa_NCGM2_S1"
from diets import load_model,constrain_exchanges,get_bounds
from overflow import get_overflow
import coralme
import cobra

In [10]:
model = coralme.io.pickle.load_pickle_me_model("./me-models/{}/MEModel-BIO-{}-ME-TS.pkl".format(org,org))
m_model = cobra.io.json.load_json_model("./agora-models/AGORA_2_01_json_renamed/{}.json".format(org))
model.optimize()
sol0 = model.solution
rlist = exchanges
# Run for all conditions
for condition in conditions:
    name = "{}_{}".format(condition,metabolite)
    dct = {k:-0.5 for k in rlist if k in model.reactions}
    constrain_exchanges(model,dct)
    model.optimize()
sol = model.solution

Set parameter Username
Academic license - for non-commercial use only - expires 2025-09-03
Read LP format model from file /tmp/tmplaolew99.lp
Reading time = 0.00 seconds
: 0 rows, 0 columns, 0 nonzeros
Read LP format model from file /tmp/tmpxl0705yp.lp
Reading time = 0.00 seconds
: 1297 rows, 3258 columns, 14028 nonzeros
Read LP format model from file /tmp/tmpfwzf8i1b.lp
Reading time = 0.00 seconds
: 1330 rows, 3256 columns, 13858 nonzeros
Iteration	 Solution to check	Solver Status
---------	------------------	-------------
        1	1.4050280687025918	Not feasible
        2	0.7025140343512959	Not feasible
        3	0.3512570171756479	Not feasible
        4	0.1756285085878240	Not feasible
        5	0.0878142542939120	Not feasible
        6	0.0439071271469560	Not feasible
        7	0.0219535635734780	Not feasible
        8	0.0109767817867390	Optimal
        9	0.0164651726801085	Optimal
       10	0.0192093681267932	Optimal
       11	0.0205814658501356	Not feasible
       12	0.01989541698

In [15]:
sol0.to_frame().loc["EX_o2(e)"]

fluxes           0.000000e+00
reduced_costs   -3.244305e-07
Name: EX_o2(e), dtype: float64

In [16]:
sol.to_frame().loc["EX_o2(e)"]

fluxes          -0.318462
reduced_costs    0.000000
Name: EX_o2(e), dtype: float64

In [8]:
NP = min([10,len(run_for)])
pool = mp.Pool(NP,maxtasksperchild=1)
pbar = tqdm(total=len(run_for),position=0,leave=True)
pbar.set_description('Building ({} threads)'.format(NP))
def collect_result(result):
    pbar.update(1)

for org in run_for:
    args = ([org])
    pool.apply_async(run,args, callback=collect_result)
pool.close()
pool.join()

Building (10 threads):   0%|          | 0/495 [00:00<?, ?it/s]

Set parameter Username
Academic license - for non-commercial use only - expires 2025-09-03
Read LP format model from file /tmp/tmplfsadoxq.lp
Reading time = 0.00 seconds
: 0 rows, 0 columns, 0 nonzeros
Set parameter Username
Academic license - for non-commercial use only - expires 2025-09-03
Read LP format model from file /tmp/tmppy33z1mt.lp
Reading time = 0.00 seconds
: 0 rows, 0 columns, 0 nonzeros
Read LP format model from file /tmp/tmpq0xxqhv1.lp
Reading time = 0.00 seconds
: 846 rows, 1800 columns, 7680 nonzeros
Set parameter Username
Academic license - for non-commercial use only - expires 2025-09-03
Read LP format model from file /tmp/tmp8dulf96v.lp
Reading time = 0.00 seconds
: 0 rows, 0 columns, 0 nonzeros
Set parameter Username
Academic license - for non-commercial use only - expires 2025-09-03
Read LP format model from file /tmp/tmpfjagmp73.lp
Reading time = 0.00 seconds
: 0 rows, 0 columns, 0 nonzeros
Set parameter Username
Academic license - for non-commercial use only - e

Building (10 threads):   0%|          | 1/495 [01:15<10:19:27, 75.24s/it]

        7	0.0658606907204340	Optimal
        6	0.0439071271469560	Optimal
Set parameter Username
Academic license - for non-commercial use only - expires 2025-09-03
Read LP format model from file /tmp/tmpph0p35zn.lp
Reading time = 0.00 seconds
: 0 rows, 0 columns, 0 nonzeros
        3	0.3512570171756479	Not feasible
Read LP format model from file /tmp/tmpe1duirsx.lp
Reading time = 0.00 seconds
: 1248 rows, 3044 columns, 13328 nonzeros
        7	0.0658606907204340	Optimal
        8	0.0768374725071730	Not feasible
Read LP format model from file /tmp/tmp_4vs1dmb.lp
Reading time = 0.00 seconds
: 1266 rows, 3042 columns, 13128 nonzeros
        4	0.1756285085878240	Optimal
        9	0.0713490816138035	Not feasible
        8	0.0768374725071730	Not feasible
        4	0.1756285085878240	Optimal
        4	0.1756285085878240	Not feasible
       10	0.0686048861671187	Not feasible
        9	0.0713490816138035	Not feasible
       11	0.0672327884437764	Not feasible
        5	0.2634427628817360	Not fe

Building (10 threads):   0%|          | 2/495 [01:31<5:34:33, 40.72s/it] 

...

Building (10 threads): 100%|█████████▉| 493/495 [2:06:19<00:24, 12.37s/it]

       16	0.4530066389722560	Optimal
       17	0.4530280779991832	Not feasible
       18	0.4530173584857196	Optimal
       19	0.4530227182424514	Optimal
       20	0.4530253981208173	Not feasible
       21	0.4530240581816344	Not feasible
       22	0.4530233882120429	Not feasible


Building (10 threads): 100%|█████████▉| 494/495 [2:06:40<00:15, 15.07s/it]

        3	0.3512570171756479	Optimal
        4	0.5268855257634719	Not feasible
        5	0.4390712714695599	Optimal
        6	0.4829783986165159	Not feasible
        7	0.4610248350430379	Not feasible
        8	0.4500480532562989	Optimal
        9	0.4555364441496684	Optimal
       10	0.4582806395963532	Not feasible
       11	0.4569085418730108	Optimal
       12	0.4575945907346820	Not feasible
       13	0.4572515663038464	Not feasible
       14	0.4570800540884286	Not feasible
       15	0.4569942979807197	Not feasible
       16	0.4569514199268652	Not feasible
       17	0.4569299808999380	Optimal
       18	0.4569407004134016	Not feasible
       19	0.4569353406566698	Not feasible
       20	0.4569326607783039	Optimal
       21	0.4569340007174868	Optimal
       22	0.4569346706870783	Optimal


Building (10 threads): 100%|██████████| 495/495 [2:08:53<00:00, 50.33s/it]